# llm-request SDK 종합 검증 노트북 (`verify_all.ipynb`)

`llm-request` SDK의 **모든 공개 함수 × 모든 분기**를 로컬 환경에서 실행·검증한다.

## 선행 조건

다음이 모두 실행 중이어야 한다 (file-manager·llm-request는 localhost, 나머지는 192.7.9.45):

| 서비스 | 포트 | 실행 방법 |
|--------|------|----------|
| Redis | 7617 | `docker compose -f infra/docker-compose.yml up -d predixy` |
| MongoDB | 27017 | `docker compose -f infra/docker-compose.yml up -d mongodb` |
| llm-handler | 8100 | `docker compose --profile handlers up -d llm-handler` |
| mdb-handler | 8300 | `docker compose --profile handlers up -d mdb-handler` |
| file-manager | 8600 | `cd packages/file-manager/service && uv run uvicorn app.main:app --port 8600` |
| llm-request | 8101 | `cd packages/llm-request/service && uv run uvicorn app.main:app --port 8101` |

`.env`에 `GEMINI_API_KEY` 필수.

## 검증 범위

- 공개 함수 6종: `llm_request`, `llm_request_async`, `llm_request_stream`, `llm_request_stream_async`, `done_check`, `fetch_by_cache_key`
- 플래그: `cache_mode`, `save_mode`, `debug`, `timings` (mdb_ttl, context_cache_ttl, http_timeout, standby_ttl)
- 파일 분기: inline 단일/복수, Files API, 부분 섞기, 20MB 한도, PDF split
- 오류 경로: 지원 안 되는 mime 타입
- 동시성: single-flight dedup

## 0. 환경 설정

In [1]:
import os
import nest_asyncio

nest_asyncio.apply()

# os.environ["LLM_REQUEST_URL"]  = "http://192.7.9.45:8101"
os.environ["LLM_REQUEST_URL"]  = "http://localhost:8101"
os.environ["FILE_MANAGER_URL"]  = "http://192.7.9.45:8600"
os.environ["MONGODB_DATABASE"] = "kw_ai_dev"  # 테스트 전용 DB

TEST_DB = os.environ["MONGODB_DATABASE"]

# SDK 기본 http_timeout이 600초로 상향됐으므로 별도 설정 불필요.
# 필요 시 요청별로 timings={"http_timeout": 1200} 같이 오버라이드.

print(f"LLM_REQUEST_URL  = {os.environ['LLM_REQUEST_URL']}")
print(f"MONGODB_DATABASE = {TEST_DB}")

LLM_REQUEST_URL  = http://localhost:8101
MONGODB_DATABASE = kw_ai_dev


## 1. 공통 유틸리티

In [2]:
import time
import uuid
from contextlib import contextmanager
from pathlib import Path


@contextmanager
def timer(label: str):
    start = time.perf_counter()
    yield
    elapsed_ms = (time.perf_counter() - start) * 1000
    print(f"  [{label}] {elapsed_ms:.0f}ms")


def print_meta(title: str, meta) -> None:
    m = meta.model_dump() if hasattr(meta, "model_dump") else meta
    print(f"--- {title} ---")
    print(f"  source        = {m.get('source')}")
    print(f"  cache_hit     = {m.get('cache_hit')}")
    print(f"  cache_key     = {str(m.get('cache_key',''))[:16]}...")
    if m.get('route_reason'):
        print(f"  route_reason  = {m['route_reason']}")
    if m.get('file_route'):
        print(f"  file_route    = {m['file_route']}")


def unique_q(base: str) -> str:
    """캐시 미스를 강제하기 위해 매번 다른 입력 생성."""
    return f"{base} [id={uuid.uuid4().hex[:8]}]"


In [3]:
from file_manager import FileManagerHandler
from llm_request import StoredFile

fm = FileManagerHandler()  # Singleton; build_metadata는 로컬 계산 (HTTP 없음)


async def build_stored_file(path) -> StoredFile:
    """로컬 파일 → file-manager 등록 → StoredFile (6 필드).

    register_or_get 이 hash 3 필드로 check_cache → miss 면 upload_file.
    bytes 는 file-manager 에만 적재되고 caller 는 이후 release 가능.
    """
    p = Path(path)
    sf_dict = await fm.register_or_get(
        bytes_data=p.read_bytes(),
        metadata=fm.build_metadata(
            file_name=p.name,
            file_mtime=int(p.stat().st_mtime),
            file_size=p.stat().st_size,
        ),
    )
    return StoredFile(**sf_dict)


## 2. 서비스 헬스체크

모든 서비스가 올라와 있는지 확인. 하나라도 FAIL이면 이후 셀은 실패.

In [4]:
import httpx

SERVICES = [
    ("llm-request",  "http://192.7.9.45:8101/health"),
    ("file-manager", "http://192.7.9.45:8600/health"),
    ("llm-handler",  "http://192.7.9.45:8100/health"),
    ("mdb-handler",  "http://192.7.9.45:8300/health"),
]

async with httpx.AsyncClient(timeout=5.0) as c:
    for name, url in SERVICES:
        try:
            r = await c.get(url)
            ok = r.status_code == 200
            print(f"  [{'OK' if ok else 'FAIL'}] {name:<13} -> {r.status_code}")
        except Exception as e:
            print(f"  [FAIL] {name:<13} -> {type(e).__name__}: {e}")

  [OK] llm-request   -> 200
  [OK] file-manager  -> 200
  [OK] llm-handler   -> 200
  [OK] mdb-handler   -> 200


## 3. `llm_request_async` — 기본 (캐시 miss → hit)

동일 입력 2회 호출해 첫 번째는 LLM, 두 번째는 MDB 캐시 히트가 되는지 확인.

In [5]:
from llm_request import llm_request_async

Q1 = unique_q("대한민국의 수도는?")

with timer("1회차 (cache miss → LLM)"):
    r1, m1 = await llm_request_async(
        {"contents": Q1, "model": "gemini-2.5-flash"},
        database=TEST_DB,
    )
print_meta("1회차", m1)
print(f"  text          = {(r1.text or '')[:80]}")

with timer("2회차 (cache hit)"):
    r2, m2 = await llm_request_async(
        {"contents": Q1, "model": "gemini-2.5-flash"},
        database=TEST_DB,
    )
print_meta("2회차", m2)

assert m2.cache_hit is True, "2회차는 캐시 히트여야 한다"
assert m1.cache_key == m2.cache_key, "동일 입력 → 동일 cache_key"
print("\n  OK: miss→hit 동작 확인")


  [1회차 (cache miss → LLM)] 1311ms
--- 1회차 ---
  source        = live
  cache_hit     = False
  cache_key     = 55caf7d891ce31bc...
  text          = 대한민국의 수도는 **서울**입니다.
  [2회차 (cache hit)] 8ms
--- 2회차 ---
  source        = cache
  cache_hit     = True
  cache_key     = 55caf7d891ce31bc...

  OK: miss→hit 동작 확인


## 4. `llm_request` (동기 래퍼)

내부에서 `asyncio.run`을 호출하지만 `nest_asyncio`로 중첩 이벤트 루프를 허용했으므로 직접 실행 가능.

In [6]:
from llm_request import llm_request

r, m = llm_request(
    {"contents": unique_q("2+2는?"), "model": "gemini-2.5-flash"},
    database=TEST_DB,
)

print_meta("sync", m)
print(f"  text          = {(r.text or '')[:80]}")

--- sync ---
  source        = live
  cache_hit     = False
  cache_key     = 410e1b17a3d0e14d...
  text          = 4


## 5. 플래그 — `cache_mode` / `save_mode` / `debug` / `timings`

- `cache_mode=False`: 캐시 조회 스킵 → 항상 LLM 호출
- `save_mode=False`: 응답 저장 안 함 → 이후 done_check 시 miss
- `debug=True`: meta에 `route_reason` 포함
- `timings={"mdb_ttl": ...}`: MDB TTL 지정

In [7]:
# cache_mode=False : 첫 호출로 저장된 결과가 있어도 무시하고 LLM 호출
Q_CM = unique_q("캐시 우회 테스트")
await llm_request_async({"contents": Q_CM, "model": "gemini-2.5-flash"}, database=TEST_DB)
r, m = await llm_request_async(
    {"contents": Q_CM, "model": "gemini-2.5-flash"},
    cache_mode=False,
    database=TEST_DB,
)
print_meta("cache_mode=False (두번째 호출도 LLM)", m)
assert m.cache_hit is False
print("  OK: cache_mode=False는 캐시 히트 안 됨")


--- cache_mode=False (두번째 호출도 LLM) ---
  source        = live
  cache_hit     = False
  cache_key     = 0681673330a543d9...
  OK: cache_mode=False는 캐시 히트 안 됨


In [8]:
# save_mode=False : 호출은 하지만 저장은 안 함 → done_check 시 miss
from llm_request import done_check

Q_SM = unique_q("저장 스킵 테스트")
r, m = await llm_request_async(
    {"contents": Q_SM, "model": "gemini-2.5-flash"},
    save_mode=False,
    database=TEST_DB,
)
print_meta("save_mode=False", m)

dc = await done_check({"contents": Q_SM, "model": "gemini-2.5-flash"}, database=TEST_DB)
print(f"  done_check.cache_hit = {dc['cache_hit']}")
assert dc["cache_hit"] is False, "save_mode=False 이후 done_check는 miss여야 한다"
print("  OK: save_mode=False는 저장 안 함")

--- save_mode=False ---
  source        = live
  cache_hit     = False
  cache_key     = c761e4498d529dea...
  done_check.cache_hit = False
  OK: save_mode=False는 저장 안 함


In [9]:
# debug=True : meta.route_reason 확인
r, m = await llm_request_async(
    {"contents": unique_q("debug 경로"), "model": "gemini-2.5-flash"},
    debug=True,
    database=TEST_DB,
)
print_meta("debug=True", m)
assert m.route_reason is not None, "debug=True일 때 route_reason 포함돼야 함"
print("  OK: route_reason 포함 확인")


--- debug=True ---
  source        = live
  cache_hit     = False
  cache_key     = 7d4a46b6d3e97409...
  route_reason  = cache_miss
  OK: route_reason 포함 확인


## 6. `done_check` + `fetch_by_cache_key`

In [10]:
from llm_request import fetch_by_cache_key

Q_F = unique_q("페치 테스트 질문")
await llm_request_async({"contents": Q_F, "model": "gemini-2.5-flash"}, database=TEST_DB)

dc = await done_check({"contents": Q_F, "model": "gemini-2.5-flash"}, database=TEST_DB)
print(f"done_check: cache_key={dc['cache_key'][:16]}... cache_hit={dc['cache_hit']}")
assert dc["cache_hit"] is True

r, m = await fetch_by_cache_key(dc["cache_key"], database=TEST_DB)
print_meta("fetch_by_cache_key", m)
print(f"  text = {(r.text or '')[:80]}")

done_check: cache_key=5c675e820eb47570... cache_hit=True
--- fetch_by_cache_key ---
  source        = cache
  cache_hit     = True
  cache_key     = 5c675e820eb47570...
  text = `fetch` API를 사용하는 코드를 테스트하는 것은 네트워크 의존성 때문에 일반적인 함수를 테스트하는 것보다 고려할 사항이 많습니다. 주로 


## 7. `llm_request_stream_async` — 스트리밍

- 1회차 (miss): SSE 청크 여러 개
- 2회차 (hit): 캐시 결과가 단일 청크로 래핑되어 동일 인터페이스로 반환

In [11]:
from llm_request import llm_request_stream_async

Q_S = unique_q("스트리밍으로 답해줘: 한국 전통 악기 3가지")

stream_1, meta_1 = await llm_request_stream_async(
    {"contents": Q_S, "model": "gemini-2.5-flash"},
    database=TEST_DB,
)
chunks_1 = []
async for ev in stream_1:
    chunks_1.append(ev)

print(f"1회차: {len(chunks_1)}개 청크 (cache_hit={meta_1['cache_hit']})")
full_text = "".join(c.get("text", "") for c in chunks_1)
print(f"  전체 텍스트: {full_text[:120]}")

stream_2, meta_2 = await llm_request_stream_async(
    {"contents": Q_S, "model": "gemini-2.5-flash"},
    database=TEST_DB,
)
chunks_2 = []
async for ev in stream_2:
    chunks_2.append(ev)

print(f"\n2회차 (cache hit): {len(chunks_2)}개 청크 (cache_hit={meta_2['cache_hit']})")

1회차: 8개 청크 (cache_hit=False)
  전체 텍스트: 자, 스트리밍 시청자 여러분! 한국 전통 악기 3가지, 바로 알려드릴게요! 라이브 느낌으로다가!

1.  **가야금 (Gayageum):**
    *   한국을 대표하는 현악기죠! 12개의 줄을 손가락으로 뜯거나 

2회차 (cache hit): 1개 청크 (cache_hit=True)


## 8. `llm_request_stream` (동기 스트리밍)

In [12]:
from llm_request import llm_request_stream

Q_SS = unique_q("동기 스트리밍으로 답해줘: 한국 5대 명절 각 2~3줄 설명")

events_1_iter, meta_1 = llm_request_stream(
    {"contents": Q_SS, "model": "gemini-2.5-flash"},
    database=TEST_DB,
)
events_1 = list(events_1_iter)
print(f"1회차 (miss): {len(events_1)}개 청크 (cache_hit={meta_1['cache_hit']})")

events_2_iter, meta_2 = llm_request_stream(
    {"contents": Q_SS, "model": "gemini-2.5-flash"},
    database=TEST_DB,
)
events_2 = list(events_2_iter)
print(f"2회차 (cache hit): {len(events_2)}개 청크 (cache_hit={meta_2['cache_hit']})")

assert len(events_1) > 1, "1회차는 SSE 청크가 여러 개여야 miss 검증 가능"
assert len(events_2) == 1, "2회차는 단일 청크로 래핑됨 (hit)"
print("  OK: miss→hit 청크 수 차이 확인")

1회차 (miss): 11개 청크 (cache_hit=False)
2회차 (cache hit): 1개 청크 (cache_hit=True)
  OK: miss→hit 청크 수 차이 확인


## 9. 파일 준비 — `build_stored_file`

테스트 파일들을 file-manager 에 등록 후 `StoredFile` 로 보관.


In [6]:
SAMPLES_DIR = Path(".").resolve()

FILES = {
    "small_pdf":  SAMPLES_DIR / "small_file.pdf",    # 612KB
    "medium_pdf": SAMPLES_DIR / "medium_file.pdf",   # 24MB
    "large_pdf":  SAMPLES_DIR / "large_file.pdf",    # 92MB
    "image":      SAMPLES_DIR / "test_image.png",    # 1.9MB
    "text":       SAMPLES_DIR / "sample.txt",        # 2KB
    "audio":      SAMPLES_DIR / "test_audio.m4a",    # 4MB (mime 미지원 테스트용)
}

inputs: dict[str, StoredFile] = {}
for key, path in FILES.items():
    if not path.exists():
        print(f"  [SKIP] {key}: {path} 없음")
        continue
    sf = await build_stored_file(path)
    inputs[key] = sf
    print(f"  [OK]   {key:<10} mime={sf.mime_type:<18} size={sf.file_size:>10,}B  file_path={sf.file_path}")


  [OK]   small_pdf  mime=application/pdf    size=   612,269B  file_path=a67e310cb97241278463f45bcebc1eac.pdf
  [OK]   medium_pdf mime=application/pdf    size=24,770,651B  file_path=6932d838e593463bb1fdfe4e4eb9f641.pdf
  [OK]   large_pdf  mime=application/pdf    size=92,716,371B  file_path=51cca2b6fd144c979648826772a2c564.pdf
  [OK]   image      mime=image/png          size= 1,914,348B  file_path=097d88a94dc9467ba8dc0bff38350721.png
  [OK]   text       mime=text/plain         size=     1,850B  file_path=4295f2892021456898a552094f60aa3f.txt
  [OK]   audio      mime=audio/mp4          size= 4,006,244B  file_path=4a979bc1002341cb89746233824428e0.m4a


## 10. Inline — PDF 단일 (612KB)

PDF 한 개가 50MB 미만이므로 inline 분기.

In [7]:
r, m = await llm_request_async(
    {"contents": unique_q("이 문서를 한 문장으로 요약해줘"), "model": "gemini-2.5-flash"},
    stored_files=[inputs["small_pdf"]],
    database=TEST_DB,
    debug=True,
)
print_meta("small_pdf (inline)", m)
print(f"  text = {(r.text or '')[:150]}")

--- small_pdf (inline) ---
  source        = live
  cache_hit     = False
  cache_key     = 32b4ba8c1d70df8f...
  route_reason  = cache_miss
  text = 이 문서는 KB금융그룹이 2024년 IFRS 지속가능성 공시기준 S1·S2에 따라 투자자 관점에서 선정한 5가지 핵심 이슈(리스크 관리, 기후위기 대응, 포용금융, 금융소비자보호, 디지털 혁신 및 기술)에 대한 거버넌스, 전략, 위험관리, 지표 및 목표를 상세히 설명하


## 11. Inline — 이미지

In [8]:
r, m = await llm_request_async(
    {"contents": unique_q("이 이미지에 무엇이 있나?"), "model": "gemini-2.5-flash"},
    stored_files=[inputs["image"]],
    database=TEST_DB,
    debug=True,
)
print_meta("image (inline)", m)
print(f"  text = {(r.text or '')[:150]}")

--- image (inline) ---
  source        = live
  cache_hit     = False
  cache_key     = 4d7509862c0c8b15...
  route_reason  = cache_miss
  text = 이 이미지는 신비롭고 고요한 숲의 야경을 보여줍니다.

**전반적인 분위기:**
*   보름달이 환하게 빛나는 어두운 숲 속 풍경으로, 안개나 옅은 연기가 깔려 몽환적이고 마법 같은 분위기를 자아냅니다.
*   푸른색 계열의 어둡고 차분한 색감이 지배적이며, 발광하는 


## 12. Inline — 텍스트

In [9]:
r, m = await llm_request_async(
    {"contents": unique_q("이 텍스트 파일의 주제는?"), "model": "gemini-2.5-flash"},
    stored_files=[inputs["text"]],
    database=TEST_DB,
    debug=True,
)
print_meta("sample.txt (inline)", m)
print(f"  text = {(r.text or '')[:150]}")

--- sample.txt (inline) ---
  source        = live
  cache_hit     = False
  cache_key     = bb63507019a50399...
  route_reason  = cache_miss
  text = 이 텍스트 파일의 주제는 **짧은 텍스트 컨텐츠** 그 자체입니다. 텍스트 내용이 "짧은 텍스트 컨텐츠입니다." 라고 반복적으로 명시하고 있기 때문입니다.


## 13. Files API — 대용량 PDF (92MB, 자동 split)

PDF 한 개가 50MB를 초과하면 Files API 분기 + PDF split. 첫 호출은 업로드 포함이라 분 단위로 걸릴 수 있음.

In [10]:
if "large_pdf" not in inputs:
    print("  [SKIP] large_file.pdf 없음")
else:
    with timer("large_pdf 첫 호출 (업로드+split+LLM)"):
        r, m = await llm_request_async(
            {"contents": unique_q("이 PDF의 제목 한 줄"), "model": "gemini-2.5-flash", "max_tokens": 100},
            stored_files=[inputs["large_pdf"]],
            database=TEST_DB,
            debug=True,
        )
    print_meta("large_pdf (Files API + split)", m)
    print(f"  text = {(r.text or '')[:150]}")

  [large_pdf 첫 호출 (업로드+split+LLM)] 84252ms
--- large_pdf (Files API + split) ---
  source        = live
  cache_hit     = False
  cache_key     = 55a9b95136525ecd...
  route_reason  = cache_miss
  text = 2025 GS건설 지속가능경영보고서


## 14. 부분 섞기 — inline + Files API 혼재

`small_pdf` + `large_pdf` → 작은 파일은 inline, 큰 파일은 Files API로 분류.

In [11]:
if "large_pdf" not in inputs:
    print("  [SKIP] large_file.pdf 없음")
else:
    r, m = await llm_request_async(
        {"contents": unique_q("두 PDF의 공통점 한 줄"), "model": "gemini-2.5-flash"},
        stored_files=[inputs["small_pdf"], inputs["large_pdf"]],
        database=TEST_DB,
        debug=True,
    )
    print_meta("small + large (혼합)", m)
    print(f"  text = {(r.text or '')[:150]}")

--- small + large (혼합) ---
  source        = live
  cache_hit     = False
  cache_key     = 96c9bebb59fb1e1c...
  route_reason  = cache_miss
  text = 두 문서 모두 한국 기업이 발행한 지속가능경영보고서로, 기업의 환경, 사회, 지배구조(ESG) 성과와 전략을 다루고 있습니다.


## 15. 이미지+PDF 혼합 — 20MB 한도 분기

이미지/오디오가 포함되면 inline 합계 한도가 20MB로 낮아진다. `image(1.9MB) + medium_pdf(24MB)` → 합계 26MB → 큰 쪽이 Files API로 승급.

In [12]:
if "medium_pdf" not in inputs:
    print("  [SKIP] medium_file.pdf 없음")
else:
    r, m = await llm_request_async(
        {"contents": unique_q("이미지와 PDF를 함께 설명"), "model": "gemini-2.5-flash"},
        stored_files=[inputs["image"], inputs["medium_pdf"]],
        database=TEST_DB,
        debug=True,
    )
    print_meta("image + medium_pdf (20MB 한도)", m)
    print(f"  text = {(r.text or '')[:150]}")

--- image + medium_pdf (20MB 한도) ---
  source        = live
  cache_hit     = False
  cache_key     = e5bd4130e35863c4...
  route_reason  = cache_miss
  text = 제공해주신 이미지와 PDF 문서는 다음과 같이 연결하여 설명할 수 있습니다.

---

**이미지 설명:**
제공된 이미지는 고요하고 신비로운 숲 풍경을 담고 있습니다. 어두운 밤하늘 아래 밝은 보름달이 숲을 비추고 있으며, 커다란 나무들이 아치형으로 길을 만들고 있습


## 16. `context_cache_ttl` — Context Cache 생성

Files API로 분류된 파일에 TTL(초)을 지정하면 Gemini Context Cache 생성. 같은 파일 조합으로 반복 호출 시 입력 토큰 90% 할인.

In [13]:
if "large_pdf" not in inputs:
    print("  [SKIP] large_file.pdf 없음")
else:
    r, m = await llm_request_async(
        {"contents": unique_q("첫 질문"), "model": "gemini-2.5-flash"},
        stored_files=[inputs["large_pdf"]],
        timings={"context_cache_ttl": 3600},
        database=TEST_DB,
        debug=True,
    )
    print_meta("context_cache 첫 호출 (생성)", m)
    assert m.source == "live", "첫 호출은 LLM 직접 호출이어야 함"

    r2, m2 = await llm_request_async(
        {"contents": unique_q("두번째 질문"), "model": "gemini-2.5-flash"},
        stored_files=[inputs["large_pdf"]],
        timings={"context_cache_ttl": 3600},
        database=TEST_DB,
        debug=True,
    )
    print_meta("context_cache 두번째 호출 (재사용)", m2)
    assert m2.source == "live", "새 질문이므로 MDB miss, LLM 직접 호출"
    print("  OK: context_cache 재사용은 서비스 로그 [route] mode=context_cache source=cached 로 확인")


--- context_cache 첫 호출 (생성) ---
  source        = live
  cache_hit     = False
  cache_key     = 9bc304b6497309e5...
  route_reason  = cache_miss
--- context_cache 두번째 호출 (재사용) ---
  source        = live
  cache_hit     = False
  cache_key     = 0dde70cc56641739...
  route_reason  = cache_miss
  OK: context_cache 재사용은 서비스 로그 [route] mode=context_cache source=cached 로 확인


## 17. 검증 에러 — 지원 안 되는 mime 타입

`test_audio.m4a` (audio/mp4 또는 audio/x-m4a) → 화이트리스트 미포함 → HTTP 422.

In [14]:
from llm_request import ServiceError

if "audio" not in inputs:
    print("  [SKIP] test_audio.m4a 없음")
else:
    print(f"  파일 mime: {inputs['audio'].mime_type}")
    try:
        r, m = await llm_request_async(
            {"contents": "전사해줘", "model": "gemini-2.5-flash"},
            stored_files=[inputs["audio"]],
            database=TEST_DB,
        )
        print("  [FAIL] 예외 발생 안 함")
    except ServiceError as e:
        print(f"  [OK] ServiceError: status={e.status_code} type={e.error_type}")
        print(f"       message: {str(e)[:120]}")

  파일 mime: audio/mp4
  [OK] ServiceError: status=422 type=unsupported_file_type
       message: [unsupported_file_type] 지원하지 않는 파일 타입: mime_type=audio/mp4, file_name=test_audio.m4a (422)


## 18. 동시성 — single-flight dedup

동일 입력 3개를 동시에 호출하면 첫 번째만 LLM 호출, 나머지는 결과를 공유받는다.

In [15]:
import asyncio

Q_D = unique_q("동시성 테스트 질문")


async def _one(i: int):
    r, m = await llm_request_async(
        {"contents": Q_D, "model": "gemini-2.5-flash"},
        database=TEST_DB,
        debug=True,
    )
    return i, (r.text or "")[:40], m


with timer("동시 3개 요청"):
    results = await asyncio.gather(_one(1), _one(2), _one(3))

for i, txt, meta in results:
    print(f"  #{i}: cache_hit={meta.cache_hit!s:<5} route={meta.route_reason!r:<30} text={txt}")

texts = {txt for _, txt, _ in results}
print(f"\n  고유 응답 수: {len(texts)} (1이면 dedup 동작)")


  [동시 3개 요청] 25657ms
  #1: cache_hit=False route='cache_miss'                   text=동시성 테스트(Concurrency Testing)는 시스템이 여러 요청
  #2: cache_hit=True  route='standby_wait_hit'             text=동시성 테스트(Concurrency Testing)는 시스템이 여러 요청
  #3: cache_hit=True  route='standby_wait_hit'             text=동시성 테스트(Concurrency Testing)는 시스템이 여러 요청

  고유 응답 수: 1 (1이면 dedup 동작)


## 정리

위 셀들이 모두 성공하면 다음이 검증된 것:

- 공개 함수 6종 전부 정상 동작
- 캐시 경로: miss → hit, 우회(cache_mode=False), 저장 스킵(save_mode=False)
- 파일 분기: inline(PDF/이미지/텍스트), Files API, 부분 섞기, 20MB 한도 분기, PDF split
- context_cache 생성/재사용
- 입력 검증(mime) fail-fast
- single-flight dedup

실패한 셀이 있으면 §2 헬스체크부터 재확인.